# Faruq-v3 FSCE-CPE breadth screening

Seed-42 validation-only breadth screen for two instance-to-instance contrastive arms: **CPE0** uses all native TAL positives; **CPE7** keeps only positives whose predicted box has IoU > 0.7 with its assigned GT. Projection is 128-D, temperature 0.2, loss weight 0.5. This is a YOLO26 transfer of FSCE's proposal-contrastive mechanism, not a literal Faster R-CNN reproduction. Test is never restored/opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/fsce-cpe-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
sys.path.insert(0, str(REPO/'src'))
os.chdir(REPO)


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(), 'Aktifkan GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
D0_CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
D0FT_REPORT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json')
ACMC1_REPORT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT/'faruq_grouped_summary.json'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file()
assert not (DATA_ROOT/'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT = PROJECT_ROOT/'experiments/faruq-v3-fsce-cpe-screening-v1'
print('GPU:', torch.cuda.get_device_name(0))
print('OUTPUT:', OUTPUT_ROOT)


In [ ]:
command = [sys.executable,'-m','pytest','-q','tests/test_fsce_cpe.py']
print('STATIC CHECK:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
command = [
    sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_fsce_cpe_screening',
    '--data-root',str(DATA_ROOT),
    '--grouped-summary',str(GROUPED_SUMMARY),
    '--d0-checkpoint',str(D0_CHECKPOINT),
    '--d0ft-report',str(D0FT_REPORT),
    '--acmc1-report',str(ACMC1_REPORT),
    '--output-root',str(OUTPUT_ROOT),
    '--seed','42','--device','0','--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.run(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(process.stdout)
if process.returncode: raise RuntimeError(f'FSCE-CPE gagal: {process.returncode}')


In [ ]:
import json, pandas as pd
from IPython.display import display
SUMMARY = OUTPUT_ROOT/'val_reports/fsce_cpe_seed42_screening.json'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False and result['test_opened'] is False
rows = [{'model':'D0FT', **result['controls']['D0FT']}, {'model':'ACMC1', **result['controls']['ACMC1']}]
rows += [{'model':name, **metrics} for name, metrics in result['candidate'].items()]
display(pd.DataFrame(rows).style.format({k:'{:.2%}' for k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
for arm in ('CPE0','CPE7'):
    print(arm, result['decisions'][arm]['decision'], result['decisions'][arm]['delta_vs_D0FT'])
print('SUMMARY:', SUMMARY)
